## 05_build_dataset

Notebook 05: build the fine tuning dataset from the platinum trajectories.

Reads platinum.run and platinum.trajectory, which notebook 04 wrote, and turns
each window into one training conversation: the system prompt, the task, and the
whole ordered exchange including every tool call and every tool result.

The unit of training is the whole trajectory, not the individual turn. A turn on
its own has no meaning, because the interesting part is the model noticing that
a tool result contradicts what it expected and changing its mind.

Output is jsonl in the chat completions shape, with tools attached, which is what
an OpenAI style fine tuning job takes and what the chat harness already speaks.

  python3 05_build_dataset.py --out /tmp/dataset.jsonl

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import random
import time
import urllib.error
import urllib.request
import uuid
from pathlib import Path


In [ ]:
import importlib

def need(module: str, package: str, why: str):
    """Import something or say exactly what to install.

    A bare `import torch` in a script that also runs as a notebook cell gives a
    ModuleNotFoundError traceback, which does not tell anyone that the fix is a
    pip line. Notebook 03 has had this shape for its dbutils shim all along, so
    these two files should not be worse.
    """
    try:
        return importlib.import_module(module)
    except ImportError:
        raise SystemExit(
            f"{module} is needed to {why}, and it is not installed.\n"
            f"  %pip install {package}\n"
            f"then restart the kernel and run this again.")


psycopg = need("psycopg", "psycopg[binary]", "read the trajectories from Lakebase")
dict_row = importlib.import_module("psycopg.rows").dict_row

# Same shape as 04. One list so the prompt a model is trained on is byte for
# byte the prompt it was asked to answer under.
import importlib.util

# __file__ exists in a script and in a Databricks notebook, but not in a plain
# IPython kernel, so fall back to the working directory rather than dying on a
# NameError when this cell runs in the notebook.
_here = Path(globals().get("__file__") or Path.cwd() / "_").resolve().parent
_spec = importlib.util.spec_from_file_location("platinum", _here / "04_gold_to_platinum.py")
platinum = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(platinum)

MASTER_SYSTEM_PROMPT = platinum.MASTER_SYSTEM_PROMPT
TASK_INSTRUCTION = platinum.TASK_INSTRUCTION
TOOLS = platinum.TOOLS
DB_HOST, DB_NAME, DB_USER = platinum.DB_HOST, platinum.DB_NAME, platinum.DB_USER

# Tool names the model emitted that are not real tools. They came from the model
# writing a shell command into the name field, and they are dropped rather than
# kept, because training on them teaches the harness to call a tool named
# `cat some/file`.
KNOWN_TOOLS = {t["function"]["name"] for t in TOOLS}


In [ ]:
def connect():
    token = os.environ.get("LAKEBASE_TOKEN", "")
    if not token:
        raise SystemExit("set LAKEBASE_TOKEN; it lives an hour, so mint a fresh one")
    return psycopg.connect(host=DB_HOST, port=5432, dbname=DB_NAME, user=DB_USER,
                           password=token, sslmode="require", connect_timeout=30,
                           row_factory=dict_row)


In [ ]:
def build_example(run: dict, steps: list[dict], windows: dict) -> dict | None:
    """One window becomes one conversation in OpenAI chat format."""
    messages: list[dict] = [
        {"role": "system", "content": MASTER_SYSTEM_PROMPT},
        {"role": "user", "content": TASK_INSTRUCTION.format(
            start=run["start_ts"], end=run["end_ts"])
         + f"\nThe ticket is {run.get('task_id') or run['window_id']}: "
           f"{run.get('title') or 'untitled'}.\n"},
    ]

    kept = dropped = 0
    # assistant steps are stored one row per tool call, sharing a turn, so group
    # them back together or the conversation reads as several separate replies
    by_turn: dict[int, list[dict]] = {}
    for step in steps:
        by_turn.setdefault(step["turn"], []).append(step)

    for turn in sorted(by_turn):
        rows = by_turn[turn]
        prose = next((r["content"] for r in rows if r["content"]), "")
        calls = []
        results = []
        for row in rows:
            name = row["tool_name"]
            if name is None:
                continue
            if name not in KNOWN_TOOLS:
                dropped += 1
                continue
            call_id = f"call_{uuid.uuid4().hex[:16]}"
            calls.append({"id": call_id, "type": "function", "function": {
                "name": name, "arguments": json.dumps(row["tool_args"] or {})}})
            # tool_call_id, not id. A tool result is matched to its call by that
            # key, and using id leaves every result orphaned, which the validator
            # in 06 then reports as 696 problems.
            results.append({"tool_call_id": call_id, "role": "tool", "name": name,
                            "content": (row["tool_result"] or "")[:12000]})
            kept += 1

        if not calls:
            if prose:
                messages.append({"role": "assistant", "content": prose})
            continue
        message = {"role": "assistant", "content": prose or None, "tool_calls": calls}
        messages.append(message)
        messages.extend(results)

    if len(messages) < 3:
        return None
    if not any(m.get("role") == "tool" for m in messages):
        return None
    # The closing handoff is stored on the run, not as a trajectory row, because
    # it carries no tool calls and rows are written per call. Without appending
    # it here every conversation ended on a tool result, and a conversation that
    # ends on a tool result teaches the model to stop mid task with no reply.
    final = (run.get("final") or "").strip()
    if final:
        messages.append({"role": "assistant", "content": final})

    if len(messages) < 3:
        return None
    if not any(m.get("role") == "tool" for m in messages):
        return None
    if messages[-1]["role"] != "assistant" or not (messages[-1].get("content") or "").strip():
        return None

    return {
        "id": f"traj_{run['window_id']}",
        "window_id": run["window_id"],
        "task_id": run.get("task_id"),
        "title": run.get("title"),
        "messages": messages,
        "metadata": {
            "turns": run["turns"], "tool_calls": kept, "dropped_tool_names": dropped,
            "tokens_in": run["tokens_in"], "tokens_out": run["tokens_out"],
            "has_final": bool(run.get("final")),
        },
    }, kept, dropped


In [ ]:
def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--out", default="/tmp/weave-dataset.jsonl")
    parser.add_argument("--val-fraction", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=17)
    parser.add_argument("--stats-only", action="store_true")
    # parse_known_args, not parse_args. __name__ is "__main__" in a notebook, so
    # the last cell runs this, and ipykernel hands argv a -f .../connection.json
    # that argparse would reject with "unrecognized arguments" and a SystemExit 2.
    args, _unknown = parser.parse_known_args()

    # %% load the trajectories
    with connect() as conn, conn.cursor() as cur:
        # platinum.run has no start_ts, the window does. Join so the ordering
        # and the timestamps both come from gold.
        cur.execute(
            "SELECT r.*, g.start_ts, g.end_ts FROM platinum.run r "
            "JOIN gold.task_window g USING (window_id) "
            "WHERE r.status = 'succeeded' ORDER BY g.start_ts")
        runs = cur.fetchall()

    print(f"runs: {len(runs)} succeeded trajectories joined to gold")

    # The table accumulated repeats from earlier test runs, so 32 windows had
    # produced 52 succeeded rows and the dataset had 31 examples over 24
    # windows. Keep the newest run per window, otherwise the same task is
    # trained on three times with three different endings.
    newest: dict[str, dict] = {}
    for run in runs:
        current = newest.get(run["window_id"])
        if current is None or (run.get("finished_at") or 0) >= (current.get("finished_at") or 0):
            newest[run["window_id"]] = run
    if len(newest) != len(runs):
        print(f"deduped {len(runs)} runs to {len(newest)} windows, "
              f"keeping the most recent each")
    runs = list(newest.values())
    runs.sort(key=lambda r: r["start_ts"])

    # build and report
    examples, kept_total, dropped_total = [], 0, 0
    for run in runs:
        if not run.get("final"):
            continue
        with connect() as conn, conn.cursor() as cur:
            cur.execute("SELECT * FROM platinum.trajectory WHERE window_id = %s "
                        "ORDER BY turn, step", (run["window_id"],))
            steps = cur.fetchall()
        built = build_example(run, steps, {})
        if not built:
            continue
        example, kept, dropped = built
        examples.append(example)
        kept_total += kept
        dropped_total += dropped

    if not examples:
        print("no usable trajectories")
        return 1

    # stats
    turns = [e["metadata"]["turns"] for e in examples]
    calls = [e["metadata"]["tool_calls"] for e in examples]
    finals = sum(1 for e in examples if e["metadata"]["has_final"])
    print(f"\nexamples      {len(examples)}")
    print(f"with handoff  {finals}")
    print(f"turns         min {min(turns)} median {sorted(turns)[len(turns)//2]} "
          f"max {max(turns)}")
    print(f"tool calls    min {min(calls)} median {sorted(calls)[len(calls)//2]} "
          f"max {max(calls)}")
    print(f"tool calls kept    {kept_total:,}   dropped as invalid {dropped_total:,}")

    roles: dict[str, int] = {}
    for example in examples:
        for message in example["messages"]:
            roles[message["role"]] = roles.get(message["role"], 0) + 1
    print(f"messages      {roles}")
    approx = sum(len(json.dumps(m)) for e in examples for m in e["messages"]) // 4
    print(f"approx tokens {approx:,}")

    if args.stats_only:
        return 0

    # split and write
    random.Random(args.seed).shuffle(examples)
    split = max(1, int(len(examples) * args.val_fraction))
    val, train = examples[:split], examples[split:]

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    for name, rows in (("train", train), ("val", val)):
        path = out.with_name(out.stem + f"-{name}.jsonl")
        with open(path, "w", encoding="utf-8") as handle:
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        print(f"wrote {path}  {len(rows)} rows")

    with open(out, "w", encoding="utf-8") as handle:
        for row in examples:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"wrote {out}  {len(examples)} rows")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 07_train_lora

Fine tune the weave agent model on the trajectory dataset.

This is the job body for 06_finetune.py. It reads the chat+tools jsonl that
05_build_dataset.py wrote, and trains a LoRA adapter on top of an instruct model,
so the result behaves like a tool using assistant rather than like a completion
engine.

Two things about the data shape, because they are the whole point:

  * A training row is one whole conversation. The system prompt names the tools,
    the user states the task, and then there are dozens of assistant and tool
    messages. Loss is applied to the assistant messages only, including the
    tool_call arguments, because those are what the model has to learn to emit.
  * Tool results are masked out. They are context, not output, and training on
    them teaches the model to hallucinate tool output.

Run it as a Databricks job on a GPU, with the dataset on a volume. It expects
the env vars the job sets: WEAVE_TRAIN_FILE, WEAVE_VAL_FILE, WEAVE_OUTPUT_DIR.

In [ ]:
from __future__ import annotations


In [ ]:
import importlib
import json
import os
import random
import time
from pathlib import Path

TRAIN_FILE = os.environ.get("WEAVE_TRAIN_FILE", "/Volumes/main/weave/datasets/train.jsonl")
VAL_FILE = os.environ.get("WEAVE_VAL_FILE", "/Volumes/main/weave/datasets/val.jsonl")
OUTPUT_DIR = Path(os.environ.get("WEAVE_OUTPUT_DIR", "/Volumes/main/weave/models/weave-agent-v1"))

# Qwen3-8B. Same family as the model that generated the trajectories, so the
# adapter starts from weights that already know this tool call format and this
# style, and it fits one 80GB GPU in bf16 with room for activations at 16k
# context. No quantisation needed, which keeps the training numerics clean.
#
# The 122B variant of this model is a 122B MoE and needs 2 to 4 GPUs even at 4
# bit, so it is not the one to iterate on. Set WEAVE_BASELINE to change it.
BASE_MODEL = os.environ.get("WEAVE_BASELINE", "Qwen/Qwen3-8B")
LOAD_IN_4BIT = os.environ.get("WEAVE_4BIT", "0") == "1"

# 18 conversations is very little data. Rank stays low on purpose: a high rank
# adapter on 18 examples memorises them rather than learning the behaviour, and
# that failure is invisible because training loss still looks fine.
LORA_R = int(os.environ.get("WEAVE_LORA_R", "8"))
LORA_ALPHA = int(os.environ.get("WEAVE_LORA_ALPHA", "16"))
EPOCHS = float(os.environ.get("WEAVE_EPOCHS", "3"))
LEARNING_RATE = float(os.environ.get("WEAVE_LR", "1e-4"))
MAX_SEQ = int(os.environ.get("WEAVE_MAX_SEQ", "16384"))
BATCH = int(os.environ.get("WEAVE_BATCH", "1"))
GRAD_ACCUM = int(os.environ.get("WEAVE_GRAD_ACCUM", "8"))
SEED = 17


In [ ]:
def need(module: str, package: str, why: str):
    """Import something or say exactly what to install.

    A bare `import torch` in a script that also runs as a notebook cell gives a
    ModuleNotFoundError traceback, which does not tell anyone that the fix is a
    pip line. Notebook 03 has had this shape for its dbutils shim all along, so
    these two files should not be worse.
    """
    try:
        return importlib.import_module(module)
    except ImportError:
        raise SystemExit(
            f"{module} is needed to {why}, and it is not installed.\n"
            f"  %pip install {package}\n"
            f"then restart the kernel and run this again.")


def load(path: str) -> list[dict]:
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def to_chat(row: dict) -> tuple[list[dict], list[bool]]:
    """Split a conversation into messages and a per-token mask of what to train on.

    Only the assistant turns get loss. The system prompt, the task, and every
    tool result are context the model reads but must not be imitated.
    """
    messages, trainable = [], []
    for message in row["messages"]:
        messages.append({
            "role": message["role"],
            "content": message.get("content") or "",
            **({"tool_calls": message["tool_calls"]} if message.get("tool_calls") else {}),
            **({"tool_call_id": message["tool_call_id"],
                "name": message.get("name")} if message.get("role") == "tool" else {}),
        })
        # an assistant turn is trainable, a tool result never is
        trainable.append(message["role"] == "assistant")
    return messages, trainable


In [ ]:
def main() -> int:
    torch = need("torch", "torch", "train on a GPU")
    Dataset = need("datasets", "datasets", "hold the conversations").Dataset
    peft = need("peft", "peft", "add the LoRA adapter")
    trl = need("trl", "trl", "run the SFT loop")
    transformers = need("transformers", "transformers", "load the base model")
    get_peft_model = peft.get_peft_model
    LoraConfig = peft.LoraConfig
    SFTTrainer = trl.SFTTrainer
    AutoModelForCausalLM = transformers.AutoModelForCausalLM
    AutoTokenizer = transformers.AutoTokenizer
    DataCollatorForCompletionOnlyLM = transformers.DataCollatorForCompletionOnlyLM
    Trainer = transformers.Trainer
    TrainingArguments = transformers.TrainingArguments
    if not torch.cuda.is_available():
        raise SystemExit(
            "no GPU on this cluster. Building the dataset works anywhere, but "
            "LoRA on an 8B model needs an 80GB GPU. Attach a GPU runtime.")

    print("=" * 68)
    print(f"weave agent fine tune\n  base   {BASE_MODEL}\n  data   {TRAIN_FILE}")
    print("=" * 68)

    # load and tokenise
    train_rows, val_rows = load(TRAIN_FILE), load(VAL_FILE)
    print(f"\n{len(train_rows)} train, {len(val_rows)} val conversations")
    print(f"  lora r={LORA_R} alpha={LORA_ALPHA} epochs={EPOCHS} lr={LEARNING_RATE}")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def to_record(row: dict) -> dict:
        messages, trainable = to_chat(row)
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False)
        ids = tokenizer(text, truncation=True, max_length=MAX_SEQ)["input_ids"]
        # mask everything before the first assistant turn, and mask nothing else
        # from the loss side. SFTTrainer applies the completion mask, so this
        # only has to be the right length.
        return {"text": text, "length": len(ids)}

    train = Dataset.from_list([to_record(r) for r in train_rows])
    val = Dataset.from_list([to_record(r) for r in val_rows]) if val_rows else None
    print(f"  longest conversation {max(r['length'] for r in train_rows)} tokens")

    # load the base model
    load_kwargs = {"device_map": "auto", "torch_dtype": torch.bfloat16}
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        print("\n  4 bit nf4")
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kwargs)
    model.config.use_cache = False
    if LOAD_IN_4BIT:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        # the projection heads, so the adapter can actually learn the new
        # behaviour rather than nudge the existing one
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()

    # completion-only masking
    # Only train on the assistant span. Getting the template markers right here
    # is fiddly, so they are discovered from the tokenizer rather than guessed.
    # Qwen's chat template is ChatML, which is not the Llama header style the
    # marker would otherwise assume, so it is taken from the template itself.
    probe = tokenizer.apply_chat_template(
        [{"role": "user", "content": "x"},
         {"role": "assistant", "content": "y"}], tokenize=False)
    marker = next((m for m in ("<|im_start|>assistant", "### Assistant:",
                               "<|start_header_id|>assistant")
                   if m in probe), "<|im_start|>assistant")
    print(f"  response template marker: {marker}")
    collator = DataCollatorForCompletionOnlyLM(
        response_template=marker, tokenizer=tokenizer)

    # run
    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=1,
        save_strategy="epoch",
        eval_strategy="epoch" if val else "no",
        bf16=True,
        gradient_checkpointing=True,
        report_to=[],
        seed=SEED,
        remove_unused_columns=False,
    )

    trainer = SFTTrainer(
        model=model, args=args, train_dataset=train, eval_dataset=val,
        data_collator=collator, tokenizer=tokenizer,
        max_seq_length=MAX_SEQ, packing=False)
    started = time.time()
    result = trainer.train()
    print(f"\ntrained in {time.time() - started:.0f}s, "
          f"train loss {result.training_loss:.4f}")

    if val is not None:
        metrics = trainer.evaluate()
        print("  eval:", {k: round(v, 4) for k, v in metrics.items()})

    # save
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(OUTPUT_DIR / "adapter"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "adapter"))
    (OUTPUT_DIR / "adapter" / "weave_manifest.json").write_text(json.dumps({
        "base_model": BASE_MODEL, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "seed": SEED,
        "train_conversations": len(train_rows), "val_conversations": len(val_rows),
        "train_loss": float(result.training_loss), "finished": time.time(),
    }, indent=2))
    print(f"\nsaved adapter to {OUTPUT_DIR / 'adapter'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## install

Run this once, then restart the kernel and Run All. `torch` is already in the Databricks ML runtime, so only the adapter libraries are installed here.

In [ ]:
%pip install -q psycopg[binary] transformers peft trl datasets accelerate
# torch is in the ML runtime. If this cluster is CPU only, the training
# cell below cannot work at all: LoRA on 8B needs an 80GB GPU.
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.device_count(), 'gpu(s)')


## train

The trainer is `07_train_lora.py`, not a cell. It needs a GPU and has to be submitted to a job, so it stays a script. The cell below runs it here for a smoke test, or a job runs it for real.

In [ ]:
import os, subprocess, sys
from pathlib import Path
here = Path(globals().get('__file__') or Path.cwd() / '_').resolve().parent
trainer = next((p for p in here.glob('*train_lora.py')), None)
if trainer is None:
    raise SystemExit(f'put 07_train_lora.py next to this notebook; looked in {here}')
env = dict(os.environ,
           WEAVE_TRAIN_FILE='/tmp/weave-ft/train.jsonl',
           WEAVE_VAL_FILE='/tmp/weave-ft/val.jsonl',
           WEAVE_OUTPUT_DIR='/tmp/weave-ft/model')
done = subprocess.run([sys.executable, str(trainer)], env=env,
                      capture_output=True, text=True)
print(done.stdout[-4000:])
print(done.stderr[-4000:])
